**데이터 셋 준비**

In [1]:
from datasets import load_dataset

dataset = load_dataset("imdb")

print(dataset)
print(dataset["train"][0])

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and

In [2]:
train_dataset = ( dataset["train"].shuffle(seed=42).select(range(5000)))
test_dataset = (dataset["test"].shuffle(seed=42).select(range(1000)))

split = train_dataset.train_test_split(test_size = 0.2, seed = 42)

train_dataset = split["train"]
val_dataset = split["test"]

In [3]:
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 4000
Validation: 1000
Test: 1000


**토큰화 하기**

In [4]:
from transformers import AutoTokenizer

model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
text = "I really enjoyed this movie!"

tokens = tokenizer.tokenize(text)

print("Tokens:")
print(tokens)

Tokens:
['i', 'really', 'enjoyed', 'this', 'movie', '!']


In [6]:
encoded = tokenizer(text)

print("Converted Tokens:")

print(
    tokenizer.convert_ids_to_tokens(
        encoded["input_ids"])
)


Converted Tokens:
['[CLS]', 'i', 'really', 'enjoyed', 'this', 'movie', '!', '[SEP]']


In [7]:
def tokenize_function(examples):

    return tokenizer(examples["text"],truncation=True,padding="max_length",
        max_length=128)

train_dataset = train_dataset.map(
    tokenize_function,
    batched=True)

val_dataset = val_dataset.map(
    tokenize_function,
    batched=True)

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [8]:
print("Tokenized Dataset:")
print(train_dataset)

print("First Example:")
print(train_dataset[0])

Tokenized Dataset:
Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 4000
})
First Example:
{'text': "This movie is just truly awful, the eye-candy that plays Ben just can make up for everything else that is wrong with this movie.<br /><br />The writer/director/producer/lead actor etc probably had a good idea to create a movie dealing with the important issues of gay marriage, family acceptance, religion, homophobia, hate crimes and just about every other issue effecting a gay man of these times, but trying to ram every issue into such a poorly conceived film does little justice to any of these causes.<br /><br />The script is poor, the casting very ordinary, but the dialogue and acting is just woeful. The homo-hating brother is played by the most camp actor and there is absolutely no chemistry between the two lead actors (I think I've seen more passion in an corn flakes ad). The acting is stiff, and the dialogue forced (a scene w

**BERT Classification Model 불러오기**

In [9]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased",
    num_labels=2)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
train_dataset = train_dataset.remove_columns(["text"])
val_dataset = val_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

train_dataset = train_dataset.rename_column(
    "label",
    "labels")

val_dataset = val_dataset.rename_column(
    "label",
    "labels")

test_dataset = test_dataset.rename_column(
    "label",
    "labels")

train_dataset.set_format("torch")
val_dataset.set_format("torch")
test_dataset.set_format("torch")

print(train_dataset[0])

{'labels': tensor(0), 'input_ids': tensor([  101,  2023,  3185,  2003,  2074,  5621,  9643,  1010,  1996,  3239,
         1011,  9485,  2008,  3248,  3841,  2074,  2064,  2191,  2039,  2005,
         2673,  2842,  2008,  2003,  3308,  2007,  2023,  3185,  1012,  1026,
         7987,  1013,  1028,  1026,  7987,  1013,  1028,  1996,  3213,  1013,
         2472,  1013,  3135,  1013,  2599,  3364,  4385,  2763,  2018,  1037,
         2204,  2801,  2000,  3443,  1037,  3185,  7149,  2007,  1996,  2590,
         3314,  1997,  5637,  3510,  1010,  2155,  9920,  1010,  4676,  1010,
        24004, 24920,  1010,  5223,  6997,  1998,  2074,  2055,  2296,  2060,
         3277,  3466,  2075,  1037,  5637,  2158,  1997,  2122,  2335,  1010,
         2021,  2667,  2000,  8223,  2296,  3277,  2046,  2107,  1037,  9996,
        10141,  2143,  2515,  2210,  3425,  2000,  2151,  1997,  2122,  5320,
         1012,  1026,  7987,  1013,  1028,  1026,  7987,  1013,  1028,  1996,
         5896,  2003,  3532, 

Fine-tuning

In [11]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)

In [12]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

model = model.to(device)

cuda


In [13]:
from torch.optim import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=2e-5
)

In [14]:
'''
model.train()

for batch in train_loader:

    # GPU로 이동
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    # Gradient 초기화
    optimizer.zero_grad()

    # Forward
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )

    # Loss
    loss = outputs.loss

    # Backpropagation
    loss.backward()

    # Weight 업데이트
    optimizer.step()

print("Training complete!")
'''

'\nmodel.train()\n\nfor batch in train_loader:\n\n    # GPU로 이동\n    input_ids = batch["input_ids"].to(device)\n    attention_mask = batch["attention_mask"].to(device)\n    labels = batch["labels"].to(device)\n\n    # Gradient 초기화\n    optimizer.zero_grad()\n\n    # Forward\n    outputs = model(\n        input_ids=input_ids,\n        attention_mask=attention_mask,\n        labels=labels\n    )\n\n    # Loss\n    loss = outputs.loss\n\n    # Backpropagation\n    loss.backward()\n\n    # Weight 업데이트\n    optimizer.step()\n\nprint("Training complete!")\n'

**Validation Loop**

In [15]:
'''
model.eval()

val_loss = 0
correct = 0
total = 0

with torch.no_grad():

    for batch in val_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        val_loss += loss.item()

        predictions = logits.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

val_loss /= len(val_loader)
val_accuracy = correct / total

print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")
'''

'\nmodel.eval()\n\nval_loss = 0\ncorrect = 0\ntotal = 0\n\nwith torch.no_grad():\n\n    for batch in val_loader:\n\n        input_ids = batch["input_ids"].to(device)\n        attention_mask = batch["attention_mask"].to(device)\n        labels = batch["labels"].to(device)\n\n        outputs = model(\n            input_ids=input_ids,\n            attention_mask=attention_mask,\n            labels=labels\n        )\n\n        loss = outputs.loss\n        logits = outputs.logits\n\n        val_loss += loss.item()\n\n        predictions = logits.argmax(dim=1)\n\n        correct += (predictions == labels).sum().item()\n        total += labels.size(0)\n\nval_loss /= len(val_loader)\nval_accuracy = correct / total\n\nprint(f"Validation Loss: {val_loss:.4f}")\nprint(f"Validation Accuracy: {val_accuracy:.4f}")\n'

In [16]:
train_losses = []
val_losses = []
val_accuracies = []

num_epochs = 3

best_val_loss = float("inf")

for epoch in range(num_epochs):

    # Training
    model.train()

    train_loss = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # Validation

    model.eval()

    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for batch in val_loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits

            val_loss += loss.item()

            predictions = logits.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    val_loss /= len(val_loader)
    val_accuracy = correct / total



    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pt")
        print("Best model saved!")
        
    print(
        f"Epoch {epoch+1}"
        f" | Train Loss: {train_loss:.4f}"
        f" | Val Loss: {val_loss:.4f}"
        f" | Val Accuracy: {val_accuracy:.4f}"
    )


Best model saved!
Epoch 1 | Train Loss: 0.4221 | Val Loss: 0.3538 | Val Accuracy: 0.8500
Best model saved!
Epoch 2 | Train Loss: 0.2285 | Val Loss: 0.3308 | Val Accuracy: 0.8650
Epoch 3 | Train Loss: 0.1132 | Val Loss: 0.4396 | Val Accuracy: 0.8630


In [17]:
model.load_state_dict(
    torch.load("best_model.pt", weights_only=True)
)

model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [18]:
test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

model.eval()

all_predictions = []
all_labels = []

test_loss = 0
correct = 0
total = 0

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        test_loss += loss.item()

        predictions = logits.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        all_predictions.extend(
            predictions.cpu().numpy())

        all_labels.extend(
            labels.cpu().numpy())

test_loss /= len(test_loader)
test_accuracy = correct / total

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

Test Loss: 0.3593
Test Accuracy: 0.8550


In [19]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
cm = confusion_matrix(
    all_labels,
    all_predictions
)

print(cm)

precision = precision_score(all_labels, all_predictions)
recall = recall_score(all_labels, all_predictions)
f1 = f1_score(all_labels, all_predictions)

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")

[[454  58]
 [ 87 401]]
Precision: 0.8736
Recall: 0.8217
F1-score: 0.8469
